# Defense LLM server (Kaggle #2) — owner: Jerush

OpenAI-compatible llama.cpp server on the T4, exposed through a **Cloudflare quick tunnel**
(`https://<random>.trycloudflare.com`: no account, no domain) and protected by an API key.
Contract: `docs/04-system-design.md` section 9.

**Before running** (Kaggle → Add-ons → Secrets, toggle it on for this notebook):
- `DEFENSE_LLM_KEY`: any long random string. Put the same value in the analyst's `.env`.

Settings: Accelerator **GPU T4**, Internet **on**. Run all cells. The tunnel cell prints the URL for
`DEFENSE_LLM_URL`. **The URL changes every run**, so update `.env` each time. The last cell keeps the session alive.

In [ ]:
# CUDA build of llama-cpp-python. Pin the exact version: PyPI's newer releases are source-only, and
# pip would silently compile them for 20+ min (CPU-only). --only-binary makes that fail fast instead.
!pip install "llama-cpp-python[server]==0.3.28" --only-binary=llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
!pip install -q huggingface_hub


In [ ]:
from huggingface_hub import hf_hub_download

# 7B instruct, Q4_K_M (~4.7 GB): fits the T4 with room for a 4k context
MODEL = hf_hub_download("bartowski/Qwen2.5-7B-Instruct-GGUF", "Qwen2.5-7B-Instruct-Q4_K_M.gguf")
MODEL

In [ ]:
import subprocess, time, requests
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
API_KEY = secrets.get_secret("DEFENSE_LLM_KEY")

server = subprocess.Popen(
    ["python", "-m", "llama_cpp.server", "--model", MODEL, "--host", "127.0.0.1", "--port", "8000",
     "--n_gpu_layers", "-1", "--n_ctx", "4096", "--api_key", API_KEY],
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(120):
    try:
        if requests.get("http://127.0.0.1:8000/v1/models", headers={"Authorization": f"Bearer {API_KEY}"}).ok:
            print("server up"); break
    except requests.ConnectionError:
        pass
    time.sleep(2)
else:
    print(open("server.log").read()[-3000:]); raise SystemExit("server did not start")

In [ ]:
# Smoke test: must return one JSON RuleProposal (section 9)
prompt = ('Reply with ONE JSON object only: {"owasp","match_type","target","pattern","params","action","description"}. '
          'A DIVERTed request: POST /api/transfer, 20 transfers in 10 s from account 35, owasp A06.')
r = requests.post("http://127.0.0.1:8000/v1/chat/completions",
                  headers={"Authorization": f"Bearer {API_KEY}"},
                  json={"messages": [{"role": "user", "content": prompt}], "temperature": 0,
                        "response_format": {"type": "json_object"}})
print(r.json()["choices"][0]["message"]["content"])

In [ ]:
# Cloudflare quick tunnel: no account or token; prints a random https://<words>.trycloudflare.com URL
import re
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        PUBLIC_URL = m.group(0); break
    time.sleep(1)
assert PUBLIC_URL, open("tunnel.log").read()[-3000:]
time.sleep(10)  # DNS for a fresh quick tunnel takes a few seconds
print("with key:   ", requests.get(f"{PUBLIC_URL}/v1/models", headers={"Authorization": f"Bearer {API_KEY}"}).status_code, "(must be 200)")
print("without key:", requests.get(f"{PUBLIC_URL}/v1/models").status_code, "(must be 401)")
print(f"\n>>> analyst .env:  LLM_MODE=kaggle  DEFENSE_LLM_URL={PUBLIC_URL}")

In [ ]:
# Keep-alive: Kaggle stops idle sessions. Interrupt this cell to shut down.
try:
    while True:
        alive = server.poll() is None and tunnel.poll() is None
        print(time.strftime("%H:%M:%S"), "server+tunnel up" if alive else "SOMETHING DIED: check server.log / tunnel.log", flush=True)
        time.sleep(60)
finally:
    server.terminate(); tunnel.terminate()